# Sección: Taller FeniCSx

Flujo de trabajo típico:
- Geometría y malla
- Espacio de funciones
- Parámetros y constantes
- Condiciones de borde
- Formulación débil
- Resolución

## 1. Definir la malla

In [28]:
import numpy as np
from dolfinx import mesh
from mpi4py import MPI

# Utilizan 2 puntos, el inicial y el final, para intervalo, rectangulo, box
# mesh.create_interval -> para interlavos lineales

comm = MPI.COMM_WORLD
cell_type = mesh.CellType.triangle

# params
L = 4
H = 1
nx = 40
ny = 10

# Se crea un rectangulo de L x H, dividido en nx x ny, con elementos triangulares
domain = mesh.create_rectangle(
    comm,
    [[0.0, 0.0], [L, H]],
    [nx, ny],
    cell_type=cell_type
)

## 2. Espacios de funciones

In [29]:
import ufl
from dolfinx import fem

# "Lagrange" es el tipo de elemento, 1 es el orden del polinomio, (2,) es la dimensión del espacio vectorial
V_space = fem.functionspace(domain, ("Lagrange", 1, (2,))) 
u = ufl.TrialFunction(V_space) # la funcion a resolver
v = ufl.TestFunction(V_space) # la funcion test

## 3. Parametros en el dominio

In [30]:
from petsc4py import PETSc

E = fem.Constant(domain, 210e9)
nu = fem.Constant(domain, np.array(0.3))
_rho = 7850.0
rho = fem.Constant(domain, _rho)
g = 9.81
b = fem.Constant(domain, [0, -_rho*g])

## 4. Condiciones de borde

### Dirichlet
prescribe desplazamientos (la función)

In [31]:
from dolfinx import default_scalar_type


# Dirichlet
def left_boundary(x):
    return np.isclose(x[0], 0.0)

left_dofs = fem.locate_dofs_geometrical(V_space, left_boundary) # se buscan los dofs que cumplen la condicion
bc1 = fem.dirichletbc(np.zeros(2, dtype=default_scalar_type), left_dofs, V_space) # se crea la condicion de frontera

def right_boundary(x):
    return np.isclose(x[0], L)

right_dofs = fem.locate_dofs_geometrical(V_space, right_boundary) # se buscan los dofs que cumplen la condicion
bc2 = fem.dirichletbc(np.ones(2, dtype=default_scalar_type), right_dofs, V_space) # se crea la condicion de frontera

## Neumann
prescribe tensiones. Se debe definir sobre las superficies de los elementos; caras

## 5. Formulación débil

En este caso para un problema de elasticidad lineal

$$
\int_\Omega \sigma(u) : \epsilon(v) d\Omega = \int_\Omega b \cdot v d\Omega + \int_{\Gamma_N} t \cdot v d \Gamma
$$

$$
a(u,v) = L(v)
$$
ese es el problema variacional creo

In [32]:
mu = E / (2 * (1 + nu))
lam = E * nu / ((1 + nu) * (1 - 2 * nu))

def epsilon(u):
    return ufl.sym(ufl.grad(u))

def sigma(u):
    return 2 * mu * epsilon(u) + lam * ufl.tr(epsilon(u)) * ufl.Identity(len(u))


In [33]:
a = ufl.inner(sigma(u), epsilon(v)) * ufl.dx
L = ufl.dot(b, v) * ufl.dx #+ ufl.dot(t, v) * ufl.ds

## 6. Resolver

In [34]:
from dolfinx.fem.petsc import LinearProblem

problem = LinearProblem(
    a, L,
    [bc1, bc2],
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu",
        "prefix": "matfree", # no se si funcion
        }
)

u_h = problem.solve()

## 7. Postproceso

In [35]:
import pyvista as pv
from dolfinx import plot

# 1. Malla no deformada
topology, cell_types, geometry = plot.vtk_mesh(domain, domain.topology.dim)
grid_undeformed = pv.UnstructuredGrid(topology, cell_types, geometry)

# 2. Malla deformada (evaluando el campo de desplazamientos u_h)
V_plot, _ = u_h.function_space.sub(0).collapse() if u_h.function_space.num_sub_spaces > 0 else (u_h.function_space, None)
plot_topology, plot_cell_types, plot_geometry = plot.vtk_mesh(V_space)
grid_deformed = pv.UnstructuredGrid(plot_topology, plot_cell_types, plot_geometry)

# Agregar los desplazamientos (completando a 3D para PyVista si es 2D)
u_values = u_h.x.array.reshape((plot_geometry.shape[0], V_space.dofmap.index_map_bs))
displacement_3d = np.zeros((plot_geometry.shape[0], 3))
displacement_3d[:, :domain.geometry.dim] = u_values

grid_deformed.point_data["Desplazamiento"] = displacement_3d

# Factor de amplificación para visualizar mejor la deformación
factor = 1e0
warped = grid_deformed.warp_by_vector("Desplazamiento", factor=factor)

# 3. Graficar ambas configuraciones superpuestas
plotter = pv.Plotter()
plotter.add_text(f"Configuración no deformada vs deformada (Escala x{factor:,.0e})", font_size=11)
plotter.add_mesh(grid_undeformed, style="wireframe", color="gray", line_width=1.5, label="No deformada")
plotter.add_mesh(warped, scalars="Desplazamiento", show_edges=True, cmap="viridis", label="Deformada")
plotter.add_legend()
plotter.view_xy()
plotter.show()

Widget(value='<iframe src="http://localhost:37107/index.html?ui=P_0x7fc8061eac50_0&reconnect=auto" class="pyvi…

# En 3D

In [36]:
import numpy as np
import ufl
from dolfinx import default_scalar_type, fem, mesh
from mpi4py import MPI
from petsc4py import PETSc

comm = MPI.COMM_WORLD
cell_type = mesh.CellType.tetrahedron

# params
L = 4
H = 1
nx = 40
ny = 10
nz = 20

dim = 3
# Se crea un rectangulo de L x H, dividido en nx x ny, con elementos triangulares
domain = mesh.create_box(
    comm,
    [[0.0, 0.0, 0.0], [L, H, H]],
    [nx, ny, nz],
    cell_type=cell_type
)


# "Lagrange" es el tipo de elemento, 1 es el orden del polinomio, (2,) es la dimensión del espacio vectorial
V_space = fem.functionspace(domain, ("Lagrange", 1, (dim,))) 
u = ufl.TrialFunction(V_space) # la funcion a resolver
v = ufl.TestFunction(V_space) # la funcion test


E = fem.Constant(domain, 210e9)
nu = fem.Constant(domain, np.array(0.3))
_rho = 7850.0
rho = fem.Constant(domain, _rho)
g = 9.81
b = fem.Constant(domain, [0, 0, -_rho*g])
t = fem.Constant(domain, [0, 0, -1000.0])


# Dirichlet
def left_boundary(x):
    return np.isclose(x[0], 0.0)

def right_boundary(x):
    return np.isclose(x[0], L)

left_dofs = fem.locate_dofs_geometrical(V_space, left_boundary) # se buscan los dofs que cumplen la condicion
bc1 = fem.dirichletbc(np.zeros(dim, dtype=default_scalar_type), left_dofs, V_space) # se crea la condicion de frontera

# Neumann
# primero se identifican las facetas de la frontera derecha, luego se crean etiquetas para esas facetas y finalmente se define la medida ds para integrar sobre esas facetas
right_facets = mesh.locate_entities_boundary(domain, dim-1, right_boundary)
facet_tags = mesh.meshtags(domain, dim-1, right_facets,np.full(len(right_facets), 1, dtype=np.int32))
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)


In [37]:
mu = E / (2 * (1 + nu))
lam = E * nu / ((1 + nu) * (1 - 2 * nu))

def epsilon(u):
    return ufl.sym(ufl.grad(u))

def sigma(u):
    return 2 * mu * epsilon(u) + lam * ufl.tr(epsilon(u)) * ufl.Identity(len(u))


In [38]:
a = ufl.inner(sigma(u), epsilon(v)) * ufl.dx
L = ufl.dot(b, v) * ufl.dx + ufl.dot(t, v) * ds(1)


In [39]:
from dolfinx.fem.petsc import LinearProblem

problem = LinearProblem(
    a, L,
    [bc1],
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu",
        "prefix": "elasticity_3d",
        }
)

u_h = problem.solve()

In [40]:
import pyvista as pv
from dolfinx import plot

# 1. Malla no deformada
topology, cell_types, geometry = plot.vtk_mesh(domain, domain.topology.dim)
grid_undeformed = pv.UnstructuredGrid(topology, cell_types, geometry)

# 2. Malla deformada (evaluando el campo de desplazamientos u_h)
V_plot, _ = u_h.function_space.sub(0).collapse() if u_h.function_space.num_sub_spaces > 0 else (u_h.function_space, None)
plot_topology, plot_cell_types, plot_geometry = plot.vtk_mesh(V_space)
grid_deformed = pv.UnstructuredGrid(plot_topology, plot_cell_types, plot_geometry)

# Agregar los desplazamientos (completando a 3D para PyVista si es 2D)
u_values = u_h.x.array.reshape((plot_geometry.shape[0], V_space.dofmap.index_map_bs))
displacement_3d = np.zeros((plot_geometry.shape[0], 3))
displacement_3d[:, :domain.geometry.dim] = u_values

grid_deformed.point_data["Desplazamiento"] = displacement_3d

# Factor de amplificación para visualizar mejor la deformación
factor = 1e4
warped = grid_deformed.warp_by_vector("Desplazamiento", factor=factor)

# 3. Graficar ambas configuraciones superpuestas
plotter = pv.Plotter()
plotter.add_text(f"Configuración no deformada vs deformada (Escala x{factor:,.0e})", font_size=11)
plotter.add_mesh(grid_undeformed, style="wireframe", color="gray", line_width=1.5, label="No deformada")
plotter.add_mesh(warped, scalars="Desplazamiento", show_edges=True, cmap="viridis", label="Deformada")
plotter.add_legend()
plotter.view_xy()
plotter.show()

Widget(value='<iframe src="http://localhost:37107/index.html?ui=P_0x7fc816859090_1&reconnect=auto" class="pyvi…